# GaiaLab governed LoRA training — Google Colab

This notebook is fail-closed and uses only the immutable `v0.8-rc2` candidate with release label `v0.8.0-rc.2`. It runs tests and governed dry-runs before loading a model, requires a CUDA GPU for training, keeps generated files outside the repository, and never uploads unless `PUSH_TO_HUB` is explicitly enabled.

**Dataset limitation:** 80 eligible records (68 train, 6 validation, 6 held-out) can validate the pipeline but cannot support reliable production-quality claims.

In [ ]:
from pathlib import Path
import datetime as dt
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile

# Safe defaults. Full training and all external writes are opt-in.
RUN_TESTS = True
RUN_DRY_RUN = True
RUN_SMOKE_TRAINING = True
RUN_FULL_TRAINING = False
RUN_EVALUATION = True
PUSH_TO_HUB = False
DOWNLOAD_OUTPUT_ARCHIVE = False

# Upload the immutable candidate ZIP directly to this Colab runtime.
# Leave blank to open the browser uploader, or use an existing /content/*.zip path.
CANDIDATE_ZIP_PATH = ""
RESUME_CHECKPOINT = ""      # Optional checkpoint-* path inside FULL_OUTPUT.

REPOSITORY_URL = "https://github.com/oluwafemidiakhoa/gaialab-naija-assistant.git"
TARGET_BRANCH = "agent/v0.8-rc2-training"
RELEASE_LABEL = "v0.8.0-rc.2"
CANDIDATE_VERSION = "v0.8-rc2"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
HUB_MODEL_ID = "mgbam/gaialab-naija-assistant-v0.8.0-rc.2-lora"
EXPECTED_AUDIT_SHA256 = "1c953505356ae8241f588f5b23f5f3ba4487e369584f789ffe26dde9b0bc8b5f"
EXPECTED_AUDIT_EVENT_COUNT = 248
EXPECTED_SOURCE_MANIFEST_SHA256 = "67bd340d2f0400222517b0b86f7f41d91839d23b11f22477e4d24b02983ffd00"
EXPECTED_CANDIDATE_SHA256 = "755165026934afc68ade34fd50610016af284cbe2cd769f3b019892e15f3189d"
EXPECTED_SPLITS = {
    "training": {"count": 68, "sha256": "92e256eb82a64be41f7d5da6df7dafc360020f7360b386751c8540fdfb927732"},
    "validation": {"count": 6, "sha256": "4bdf6777660c61e3e33aad2f83896bacc8d51e567a3fcc47935af39f68d444ec"},
    "held_out_benchmark": {"count": 6, "sha256": "2ecd247a9c8865b76346748f927b21939a81e9c4cc2607d6622570221cdc2748"},
}

REPO_DIR = Path("/content/gaialab-naija-assistant")
AUDIT_ROOT = REPO_DIR / "evaluation/review_audit"
DATA_ROOT = Path("/content/gaialab-data")
OUTPUT_ROOT = Path("/content/gaialab-output")
FULL_OUTPUT = OUTPUT_ROOT / "training-v0.8.0-rc.2"

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if not IN_COLAB:
    raise RuntimeError("This notebook must run in Google Colab")

DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STATUS = {
    "governance": "not_run",
    "tests": "not_run",
    "smoke_training": "not_run",
    "full_training": "disabled" if not RUN_FULL_TRAINING else "not_run",
    "evaluation": "not_run",
    "hub_upload": "disabled" if not PUSH_TO_HUB else "not_run",
}
print("Google Colab detected:", IN_COLAB)
print("Repository:", REPO_DIR)
print("Governed data root:", DATA_ROOT)
print("Generated output root:", OUTPUT_ROOT)

In [ ]:
def run_command(arguments, *, cwd=None, env=None):
    """Print complete stdout/stderr, then fail with the command's exit code."""
    command = [str(value) for value in arguments]
    print("\n+", " ".join(command))
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    print("--- stdout ---")
    print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    print("--- stderr ---", file=sys.stderr)
    print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_versioned_json(path, payload):
    """Never overwrite a report; reuse identical content or create a timestamped revision."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + "\n"
    if not path.exists():
        path.write_text(text, encoding="utf-8")
        return path
    if path.read_text(encoding="utf-8") == text:
        return path
    stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    revision = path.with_name(f"{path.stem}.{stamp}{path.suffix}")
    if revision.exists():
        raise RuntimeError(f"Report revision already exists: {revision}")
    revision.write_text(text, encoding="utf-8")
    return revision

def package_versions(names):
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

def require_cuda(context):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(f"CUDA GPU required for {context}; select Runtime > Change runtime type > GPU")
    return torch.cuda.get_device_name(0)

def completed_manifest(output_dir, allowed_statuses, *, allow_incomplete=False):
    paths = sorted(Path(output_dir).glob("training_manifest*.json"), key=lambda path: path.stat().st_mtime)
    if not paths:
        return None
    completed = [read_json(path) for path in paths if read_json(path).get("training_completion_status") in allowed_statuses]
    if completed:
        return completed[-1]
    if allow_incomplete:
        return None
    raise RuntimeError(f"Existing run is incomplete or invalid: {paths[-1]}")

def extract_zip_safely(zip_path, destination):
    """Extract Windows or POSIX ZIP paths without traversal or silent overwrite."""
    destination = Path(destination).resolve()
    if destination.exists():
        raise RuntimeError(f"Refusing to overwrite extraction directory: {destination}")
    destination.mkdir(parents=True)
    normalized_names = set()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            normalized = member.filename.replace("\\", "/").lstrip("/")
            if not normalized or normalized.endswith("/"):
                if normalized:
                    (destination / normalized).mkdir(parents=True, exist_ok=True)
                continue
            if normalized in normalized_names:
                raise RuntimeError(f"Duplicate normalized ZIP member: {normalized}")
            normalized_names.add(normalized)
            target = (destination / normalized).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}") from exc
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("xb") as output:
                shutil.copyfileobj(source, output)
    if not normalized_names:
        raise RuntimeError("Candidate ZIP contains no files")

print("Command and integrity helpers ready")

In [ ]:
# Clone once; subsequent runs update only a clean Colab checkout.
if not (REPO_DIR / ".git").is_dir():
    run_command(["git", "clone", REPOSITORY_URL, REPO_DIR])
else:
    dirty = run_command(["git", "status", "--porcelain"], cwd=REPO_DIR).stdout.strip()
    if dirty:
        raise RuntimeError("Repository checkout is dirty; refusing to replace local changes")
    run_command(["git", "fetch", "origin", TARGET_BRANCH], cwd=REPO_DIR)
run_command(["git", "checkout", TARGET_BRANCH], cwd=REPO_DIR)
run_command(["git", "pull", "--ff-only", "origin", TARGET_BRANCH], cwd=REPO_DIR)
GIT_SHA = run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
CHECKED_OUT_BRANCH = run_command(["git", "branch", "--show-current"], cwd=REPO_DIR).stdout.strip()
if CHECKED_OUT_BRANCH != TARGET_BRANCH:
    raise RuntimeError(f"Wrong branch: {CHECKED_OUT_BRANCH}")
print("Checked-out commit SHA:", GIT_SHA)

In [ ]:
# Inspect Colab's PyTorch/CUDA environment before installing anything.
import torch
TORCH_BEFORE = torch.__version__
TORCH_FILE_BEFORE = torch.__file__
print("Python version:", platform.python_version())
print("torch version:", TORCH_BEFORE)
print("torch file:", TORCH_FILE_BEFORE)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

# requirements-colab.txt intentionally contains no torch, torchvision, or torchaudio.
requirement_lines = (REPO_DIR / "requirements-colab.txt").read_text(encoding="utf-8").splitlines()
distributions = {
    line.split("#", 1)[0].strip().split("==", 1)[0].casefold()
    for line in requirement_lines if line.split("#", 1)[0].strip()
}
for forbidden in ("torch", "torchvision", "torchaudio"):
    if forbidden in distributions:
        raise RuntimeError(f"Forbidden PyTorch package in requirements-colab.txt: {forbidden}")
run_command([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "--upgrade-strategy", "only-if-needed", "-r", REPO_DIR / "requirements-colab.txt",
])
torch_distribution = importlib.metadata.version("torch")
if torch_distribution != TORCH_BEFORE.split("+")[0]:
    raise RuntimeError("The dependency install changed Colab's preinstalled PyTorch")
print("\nDependency installation complete.")
print("If any Hugging Face package was already imported, use Runtime > Restart session now,")
print("then rerun this notebook from the first cell. Do not reinstall PyTorch.")

In [ ]:
# Post-install/post-restart verification.
import torch
if not getattr(torch, "__version__", None):
    raise RuntimeError("torch.__version__ is unavailable")
if torch.__file__ is None:
    raise RuntimeError("torch.__file__ is unavailable")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; select a Colab GPU runtime")
if torch.__version__.split("+")[0] != TORCH_BEFORE.split("+")[0]:
    raise RuntimeError("Colab's preinstalled PyTorch version changed")

import accelerate
import datasets
import huggingface_hub
import peft
import transformers

PINNED_PACKAGES = package_versions([
    "torch", "transformers", "peft", "accelerate", "datasets",
    "huggingface_hub", "tokenizers", "safetensors", "sentencepiece", "PyYAML",
])
print(json.dumps(PINNED_PACKAGES, indent=2, sort_keys=True))
print("CUDA GPU verified:", torch.cuda.get_device_name(0))

In [ ]:
# Acquire the immutable candidate from a direct browser ZIP upload. Google Drive is not used.
candidate_link = DATA_ROOT / CANDIDATE_VERSION
zip_path = Path(CANDIDATE_ZIP_PATH) if CANDIDATE_ZIP_PATH else None
if zip_path is None:
    from google.colab import files
    print("Select the immutable v0.8-rc2 candidate ZIP from your computer.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one immutable candidate ZIP")
    name, content = next(iter(uploaded.items()))
    zip_path = DATA_ROOT / "uploads" / Path(name).name
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists() and sha256_file(zip_path) != hashlib.sha256(content).hexdigest():
        raise RuntimeError(f"Refusing to overwrite a different uploaded ZIP: {zip_path}")
    if not zip_path.exists():
        zip_path.write_bytes(content)
if not zip_path.is_file() or not zipfile.is_zipfile(zip_path):
    raise RuntimeError(f"Candidate ZIP is missing or invalid: {zip_path}")
extraction = DATA_ROOT / "extracted-v2" / sha256_file(zip_path)
if not extraction.exists():
    extract_zip_safely(zip_path, extraction)
manifests = [path for path in extraction.rglob("*") if path.is_file() and path.name.lower() == "release_candidate_manifest.json"]
if len(manifests) != 1:
    archive_entries = sorted(path.relative_to(extraction).as_posix() for path in extraction.rglob("*") if path.is_file())
    preview = "\n".join(archive_entries[:50]) or "<no files extracted>"
    raise RuntimeError(
        f"Candidate ZIP must contain exactly one release_candidate_manifest.json; "
        f"found {len(manifests)}. Extracted files:\n{preview}"
    )
source = manifests[0].parent.resolve()
for path in sorted(source.rglob("*"), reverse=True):
    path.chmod(0o555 if path.is_dir() else 0o444)
source.chmod(0o555)
if candidate_link.exists() or candidate_link.is_symlink():
    if candidate_link.resolve() != source:
        raise RuntimeError(f"Candidate link already targets another directory: {candidate_link}")
else:
    candidate_link.symlink_to(source, target_is_directory=True)
CANDIDATE_DIR = candidate_link

TRAIN_FILE = CANDIDATE_DIR / "training.jsonl"
VALIDATION_FILE = CANDIDATE_DIR / "validation.jsonl"
BENCHMARK_FILE = CANDIDATE_DIR / "held_out_benchmark.jsonl"
MANIFEST_FILE = CANDIDATE_DIR / "release_candidate_manifest.json"
ELIGIBILITY_FILE = CANDIDATE_DIR / "eligibility_report.json"
for required in (TRAIN_FILE, VALIDATION_FILE, BENCHMARK_FILE, MANIFEST_FILE, ELIGIBILITY_FILE):
    if not required.is_file():
        raise RuntimeError(f"Candidate file missing: {required}")
print("Candidate mounted at:", CANDIDATE_DIR)
print("Candidate manifest SHA-256:", sha256_file(MANIFEST_FILE))

In [ ]:
# Run the full suite without hiding collection errors.
if RUN_TESTS:
    run_command([sys.executable, "-m", "pytest", "-vv", "--tb=short", "-ra"], cwd=REPO_DIR)
    STATUS["tests"] = "passed"
else:
    STATUS["tests"] = "disabled_by_user"
print("Test status:", STATUS["tests"])

In [ ]:
# Fail-closed governed dry-runs occur before any model is loaded.
manifest = read_json(MANIFEST_FILE)
if manifest.get("target_version") != CANDIDATE_VERSION:
    raise RuntimeError("Candidate version binding failed")
if manifest.get("source_version") != "v0.8-draft":
    raise RuntimeError("Unexpected source release")
candidate_payload = {key: value for key, value in manifest.items() if key != "release_candidate_sha256"}
candidate_hash = hashlib.sha256(json.dumps(candidate_payload, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
if candidate_hash != EXPECTED_CANDIDATE_SHA256 or manifest.get("release_candidate_sha256") != EXPECTED_CANDIDATE_SHA256:
    raise RuntimeError("Release-candidate SHA-256 mismatch")
if manifest.get("human_audit_sha256") != EXPECTED_AUDIT_SHA256 or manifest.get("human_audit_event_count") != EXPECTED_AUDIT_EVENT_COUNT:
    raise RuntimeError("Authoritative human-audit binding mismatch")
source_manifest = REPO_DIR / "data/releases/v0.8-draft/dataset_manifest.json"
if sha256_file(source_manifest) != EXPECTED_SOURCE_MANIFEST_SHA256 or manifest.get("source_manifest_sha256") != EXPECTED_SOURCE_MANIFEST_SHA256:
    raise RuntimeError("Source-manifest SHA-256 mismatch")
split_counts = {name: manifest.get("splits", {}).get(name, {}).get("count", 0) for name in ("training", "validation", "held_out_benchmark")}
if split_counts != {name: value["count"] for name, value in EXPECTED_SPLITS.items()}:
    raise RuntimeError(f"Governed split counts mismatch: {split_counts}")
if manifest.get("eligible_count") != 80 or manifest.get("excluded_count") != 40:
    raise RuntimeError("Eligible count does not match governed split counts")
split_files = {"training": TRAIN_FILE, "validation": VALIDATION_FILE, "held_out_benchmark": BENCHMARK_FILE}
split_rows = {}
for name, path in split_files.items():
    if sha256_file(path) != EXPECTED_SPLITS[name]["sha256"] or manifest["splits"][name]["sha256"] != EXPECTED_SPLITS[name]["sha256"]:
        raise RuntimeError(f"{name} SHA-256 mismatch")
    split_rows[name] = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(split_rows[name]) != EXPECTED_SPLITS[name]["count"]:
        raise RuntimeError(f"{name} physical record count mismatch")
def prompt_key(row):
    user = row["messages"][1]["content"].casefold()
    return " ".join("".join(ch if ch.isalnum() else " " for ch in user).split())
for left, right in (("training", "validation"), ("training", "held_out_benchmark"), ("validation", "held_out_benchmark")):
    for key in (lambda row: row["id"], lambda row: row["example_sha256"], prompt_key):
        if {key(row) for row in split_rows[left]} & {key(row) for row in split_rows[right]}:
            raise RuntimeError(f"Split leakage detected: {left}/{right}")

preflight = OUTPUT_ROOT / "preflight" / sha256_file(MANIFEST_FILE)[:16]
training_preflight = preflight / "training"
benchmark_preflight = preflight / "benchmark"
if RUN_DRY_RUN:
    existing = completed_manifest(training_preflight, {"dry_run_validated"})
    if existing is None:
        run_command([
            sys.executable, REPO_DIR / "scripts/train_governed_lora.py",
            "--config", REPO_DIR / "configs/training/v0.8.0-rc.2.yaml",
            "--release-version", RELEASE_LABEL,
            "--train-file", TRAIN_FILE,
            "--validation-file", VALIDATION_FILE,
            "--held-out-benchmark-file", BENCHMARK_FILE,
            "--source-manifest-file", source_manifest,
            "--output-dir", training_preflight,
            "--dry-run",
        ], cwd=REPO_DIR)
        existing = completed_manifest(training_preflight, {"dry_run_validated"})
    if existing.get("governance", {}).get("candidate_version") != CANDIDATE_VERSION:
        raise RuntimeError("Training dry-run candidate binding failed")
    if existing.get("release_version") != RELEASE_LABEL:
        raise RuntimeError("Training dry-run release-label binding failed")
    if existing.get("inputs", {}).get("training", {}).get("sha256") != sha256_file(TRAIN_FILE):
        raise RuntimeError("Training dry-run input hash mismatch")
    if existing.get("inputs", {}).get("validation", {}).get("sha256") != sha256_file(VALIDATION_FILE):
        raise RuntimeError("Validation dry-run input hash mismatch")

    benchmark_summary = benchmark_preflight / "evaluation_summary.json"
    if not benchmark_summary.is_file():
        run_command([
            sys.executable, REPO_DIR / "scripts/evaluate_governed_adapter.py",
            "--release-version", RELEASE_LABEL,
            "--adapter-dir", "/content/not-loaded-during-dry-run",
            "--training-file", TRAIN_FILE,
            "--evaluation-file", BENCHMARK_FILE,
            "--output-dir", benchmark_preflight,
            "--dry-run",
        ], cwd=REPO_DIR)
    benchmark_result = read_json(benchmark_summary)
    if benchmark_result.get("status") != "dry_run_validated":
        raise RuntimeError("Benchmark dry-run did not validate")
    if benchmark_result.get("evaluation_sha256") != sha256_file(BENCHMARK_FILE):
        raise RuntimeError("Benchmark dry-run input hash mismatch")
    STATUS["governance"] = "passed"
else:
    STATUS["governance"] = "disabled_by_user"

if STATUS["governance"] != "passed":
    raise RuntimeError("Governance dry-run is required before model loading")
print("Governance status:", STATUS["governance"])
print("Split counts:", split_counts)

In [ ]:
# Record the verified environment and immutable input hashes before training.
import torch
DATASET_HASHES = {
    "release_candidate_manifest.json": sha256_file(MANIFEST_FILE),
    "eligibility_report.json": sha256_file(ELIGIBILITY_FILE),
    "training.jsonl": sha256_file(TRAIN_FILE),
    "validation.jsonl": sha256_file(VALIDATION_FILE),
    "held_out_benchmark.jsonl": sha256_file(BENCHMARK_FILE),
}
environment_report = {
    "schema_version": "1.0",
    "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "colab_detected": IN_COLAB,
    "python_version": platform.python_version(),
    "git_branch": CHECKED_OUT_BRANCH,
    "git_commit_sha": GIT_SHA,
    "torch_version": torch.__version__,
    "torch_file": torch.__file__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0),
    "packages": PINNED_PACKAGES,
    "release_label": RELEASE_LABEL,
    "candidate_version": CANDIDATE_VERSION,
    "dataset_hashes": DATASET_HASHES,
    "source_candidate_path": str(CANDIDATE_DIR.resolve()),
    "audit_root_repository_path": str(AUDIT_ROOT),
    "output_root": str(OUTPUT_ROOT),
}
ENVIRONMENT_REPORT_PATH = write_versioned_json(OUTPUT_ROOT / "environment_report.json", environment_report)
print("Environment report:", ENVIRONMENT_REPORT_PATH)

In [ ]:
# Hugging Face authentication is needed only for an explicit upload.
HF_TOKEN = None
if PUSH_TO_HUB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("Set the HF_TOKEN Colab secret before enabling PUSH_TO_HUB")
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face authentication completed; token value was not displayed")
else:
    print("Hugging Face secret not read because PUSH_TO_HUB=False.")
    print("A configured HF_TOKEN does not enable upload automatically.")
    print("To upload after a governed full run, set RUN_FULL_TRAINING=True and PUSH_TO_HUB=True, then Run all.")

In [ ]:
# Five-step CUDA smoke training is mandatory before full training.
SMOKE_OUTPUT = OUTPUT_ROOT / "smoke-v0.8.0-rc.2"
if RUN_SMOKE_TRAINING:
    if STATUS["tests"] != "passed" or STATUS["governance"] != "passed":
        raise RuntimeError("Tests and governance must pass before smoke training")
    require_cuda("smoke training")
    smoke_manifest = completed_manifest(SMOKE_OUTPUT, {"smoke_test_completed"})
    if smoke_manifest is None:
        run_command([
            sys.executable, REPO_DIR / "scripts/train_governed_lora.py",
            "--config", REPO_DIR / "configs/training/v0.8.0-rc.2.yaml",
            "--train-file", TRAIN_FILE,
            "--validation-file", VALIDATION_FILE,
            "--held-out-benchmark-file", BENCHMARK_FILE,
            "--source-manifest-file", source_manifest,
            "--output-dir", SMOKE_OUTPUT,
            "--smoke-test",
        ], cwd=REPO_DIR)
        smoke_manifest = completed_manifest(SMOKE_OUTPUT, {"smoke_test_completed"})
    if smoke_manifest.get("inputs", {}).get("training", {}).get("sha256") != DATASET_HASHES["training.jsonl"]:
        raise RuntimeError("Existing smoke run used a different training split")
    if smoke_manifest.get("inputs", {}).get("validation", {}).get("sha256") != DATASET_HASHES["validation.jsonl"]:
        raise RuntimeError("Existing smoke run used a different validation split")
    STATUS["smoke_training"] = "passed"
else:
    STATUS["smoke_training"] = "disabled_by_user"
print("Smoke-training status:", STATUS["smoke_training"])

In [ ]:
# Full training remains disabled by default and cannot bypass the smoke gate.
training_manifest = None
if RUN_FULL_TRAINING:
    if STATUS["smoke_training"] != "passed":
        raise RuntimeError("A successful five-step CUDA smoke run is required")
    require_cuda("full training")
    training_manifest = completed_manifest(FULL_OUTPUT, {"training_completed"}, allow_incomplete=bool(RESUME_CHECKPOINT))
    if training_manifest is None:
        command = [
            sys.executable, REPO_DIR / "scripts/train_governed_lora.py",
            "--config", REPO_DIR / "configs/training/v0.8.0-rc.2.yaml",
            "--train-file", TRAIN_FILE,
            "--validation-file", VALIDATION_FILE,
            "--held-out-benchmark-file", BENCHMARK_FILE,
            "--source-manifest-file", source_manifest,
            "--output-dir", FULL_OUTPUT,
        ]
        if RESUME_CHECKPOINT:
            command.extend(["--resume-from-checkpoint", RESUME_CHECKPOINT])
        run_command(command, cwd=REPO_DIR)
        training_manifest = completed_manifest(FULL_OUTPUT, {"training_completed"})
    if training_manifest.get("inputs", {}).get("training", {}).get("sha256") != DATASET_HASHES["training.jsonl"]:
        raise RuntimeError("Existing full run used a different training split")
    if training_manifest.get("inputs", {}).get("validation", {}).get("sha256") != DATASET_HASHES["validation.jsonl"]:
        raise RuntimeError("Existing full run used a different validation split")
    STATUS["full_training"] = "passed"
else:
    print("Full training is disabled. Set RUN_FULL_TRAINING=True only after reviewing smoke evidence.")
print("Full-training status:", STATUS["full_training"])

In [ ]:
# Evaluate only validation and held-out benchmark splits.
adapter_dir = (FULL_OUTPUT if STATUS["full_training"] == "passed" else SMOKE_OUTPUT) / "adapter"
evaluation_summaries = {}
if RUN_EVALUATION:
    if STATUS["smoke_training"] != "passed" and STATUS["full_training"] != "passed":
        raise RuntimeError("No governed adapter is available for evaluation")
    require_cuda("evaluation")
    adapter_identity = sha256_file((adapter_dir.parent / "training_manifest.json"))[:16]
    for split_name, split_path in (("validation", VALIDATION_FILE), ("held_out_benchmark", BENCHMARK_FILE)):
        destination = OUTPUT_ROOT / "evaluations" / adapter_identity / split_name
        summary_path = destination / "evaluation_summary.json"
        if not summary_path.is_file():
            run_command([
                sys.executable, REPO_DIR / "scripts/evaluate_governed_adapter.py",
                "--release-version", RELEASE_LABEL,
                "--base-model", BASE_MODEL,
                "--adapter-dir", adapter_dir,
                "--training-file", TRAIN_FILE,
                "--evaluation-file", split_path,
                "--output-dir", destination,
            ], cwd=REPO_DIR)
        summary = read_json(summary_path)
        if summary.get("status") != "evaluation_completed":
            raise RuntimeError(f"Evaluation did not complete for {split_name}")
        if summary.get("evaluation_sha256") != sha256_file(split_path):
            raise RuntimeError(f"Evaluation input hash mismatch for {split_name}")
        if not (destination / f"predictions.{split_name}.jsonl").is_file():
            raise RuntimeError(f"Predictions missing for {split_name}")
        evaluation_summaries[split_name] = summary
    VALIDATION_METRICS_PATH = write_versioned_json(OUTPUT_ROOT / "validation_metrics.json", evaluation_summaries["validation"])
    HELD_OUT_METRICS_PATH = write_versioned_json(OUTPUT_ROOT / "held_out_metrics.json", evaluation_summaries["held_out_benchmark"])
    EVALUATION_METRICS_PATH = write_versioned_json(OUTPUT_ROOT / "evaluation_metrics.json", evaluation_summaries)
    STATUS["evaluation"] = "passed"
else:
    STATUS["evaluation"] = "disabled_by_user"
print("Evaluation status:", STATUS["evaluation"])

In [ ]:
# Upload only after explicit opt-in and successful full training/evaluation.
if PUSH_TO_HUB:
    if STATUS["full_training"] != "passed" or STATUS["evaluation"] != "passed":
        raise RuntimeError("Full training and evaluation must pass before upload")
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HUB_MODEL_ID, repo_type="model", exist_ok=True, private=True)
    api.upload_folder(repo_id=HUB_MODEL_ID, repo_type="model", folder_path=FULL_OUTPUT / "adapter")
    STATUS["hub_upload"] = "passed"
    print("Uploaded adapter to:", HUB_MODEL_ID)
else:
    STATUS["hub_upload"] = "disabled_by_user"
    print("Upload skipped safely: PUSH_TO_HUB=False.")
    print("Your HF_TOKEN may be configured, but possession of a token is not upload consent.")
    print("Set RUN_FULL_TRAINING=True and PUSH_TO_HUB=True in the configuration cell, then Run all.")

In [ ]:
# Prove immutable inputs are unchanged, then produce output hashes and a reproducibility report.
current_dataset_hashes = {
    "release_candidate_manifest.json": sha256_file(MANIFEST_FILE),
    "eligibility_report.json": sha256_file(ELIGIBILITY_FILE),
    "training.jsonl": sha256_file(TRAIN_FILE),
    "validation.jsonl": sha256_file(VALIDATION_FILE),
    "held_out_benchmark.jsonl": sha256_file(BENCHMARK_FILE),
}
if current_dataset_hashes != DATASET_HASHES:
    raise RuntimeError("Immutable candidate content changed during the run")
active_run = FULL_OUTPUT if STATUS["full_training"] == "passed" else SMOKE_OUTPUT
active_manifest_path = None
active_manifest = {}
for candidate_path in sorted(active_run.glob("training_manifest*.json"), key=lambda path: path.stat().st_mtime, reverse=True):
    candidate_manifest = read_json(candidate_path)
    if candidate_manifest.get("training_completion_status") in {"training_completed", "smoke_test_completed"}:
        active_manifest_path = candidate_path
        active_manifest = candidate_manifest
        break
output_hashes = {}
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and "reproducibility_report" not in path.name:
        output_hashes[str(path.relative_to(OUTPUT_ROOT))] = sha256_file(path)
reproducibility_report = {
    "schema_version": "1.0",
    "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "release_label": RELEASE_LABEL,
    "candidate_version": CANDIDATE_VERSION,
    "governance_status": STATUS["governance"],
    "git_commit_sha": GIT_SHA,
    "git_branch": CHECKED_OUT_BRANCH,
    "base_model": BASE_MODEL,
    "gpu_name": environment_report["gpu_name"],
    "package_versions": PINNED_PACKAGES,
    "dataset_hashes": DATASET_HASHES,
    "lora_configuration": active_manifest.get("lora"),
    "training_duration_seconds": active_manifest.get("metrics", {}).get("train_runtime"),
    "training_manifest": str(active_manifest_path) if active_manifest_path else None,
    "training_manifest_sha256": sha256_file(active_manifest_path) if active_manifest_path else None,
    "evaluation_metrics": evaluation_summaries,
    "output_hashes": output_hashes,
    "statuses": dict(STATUS),
    "limitations": "80 governed records validate the pipeline only; they do not support reliable production-quality claims.",
}
REPRODUCIBILITY_REPORT_PATH = write_versioned_json(OUTPUT_ROOT / "reproducibility_report.json", reproducibility_report)
print("Reproducibility report:", REPRODUCIBILITY_REPORT_PATH)

In [ ]:
# Optional browser download. This does not use or mount Google Drive.
DOWNLOADED_ARCHIVE = None
if DOWNLOAD_OUTPUT_ARCHIVE:
    from google.colab import files
    stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive_base = OUTPUT_ROOT.parent / f"gaialab-output-{GIT_SHA[:12]}-{stamp}"
    DOWNLOADED_ARCHIVE = archive_base.with_suffix(".zip")
    if DOWNLOADED_ARCHIVE.exists():
        raise RuntimeError(f"Refusing to overwrite output archive: {DOWNLOADED_ARCHIVE}")
    shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_ROOT)
    print("Downloading output archive:", DOWNLOADED_ARCHIVE)
    files.download(str(DOWNLOADED_ARCHIVE))
else:
    print("Output download disabled; set DOWNLOAD_OUTPUT_ARCHIVE=True when ready")

In [ ]:
# Final governed run summary.
final_summary = {
    "governance_status": STATUS["governance"],
    "test_status": STATUS["tests"],
    "smoke_training_status": STATUS["smoke_training"],
    "full_training_status": STATUS["full_training"],
    "evaluation_status": STATUS["evaluation"],
    "hugging_face_upload_status": STATUS["hub_upload"],
    "output_directory": str(OUTPUT_ROOT),
    "downloaded_archive": str(DOWNLOADED_ARCHIVE) if DOWNLOADED_ARCHIVE else None,
    "git_commit_sha": GIT_SHA,
    "candidate_version": CANDIDATE_VERSION,
    "release_label": RELEASE_LABEL,
}
FINAL_SUMMARY_PATH = write_versioned_json(OUTPUT_ROOT / "final_summary.json", final_summary)
print(json.dumps(final_summary, indent=2, sort_keys=True))
print("Final summary report:", FINAL_SUMMARY_PATH)